In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from sentence_transformers import CrossEncoder
import wandb
from kaggle_secrets import UserSecretsClient




In [19]:
device = 0 if torch.cuda.is_available() else -1
print("GPU:", torch.cuda.is_available())

def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

GPU: True


In [20]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [21]:
# Map@3 utility
def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

In [22]:
# DeBERTa Zero-Shot 

print("Loading DeBERTa...")
deberta = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)

def get_deberta_scores(row):
    options = {opt: str(row[opt]) for opt in ['A','B','C','D','E']}
    result = deberta(
        str(row['prompt']),
        candidate_labels=list(options.values()),
        hypothesis_template="The correct answer to the question is: {}"
    )
    label_to_opt = {v: k for k, v in options.items()}
    # Return dict of {A: score, B: score, ...}
    return {label_to_opt[label]: score 
            for label, score in zip(result['labels'], result['scores'])}

Loading DeBERTa...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# MiniLM Cross-Encoder

print("Loading MiniLM Cross-Encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

def get_crossencoder_scores(row):
    prompt = str(row['prompt'])
    options = {opt: str(row[opt]) for opt in ['A','B','C','D','E']}
    
    # Cross-encoder scores each (prompt, option) pair together
    pairs = [[prompt, text] for text in options.values()]
    scores = cross_encoder.predict(pairs)
    
    return {opt: float(score) 
            for opt, score in zip(options.keys(), scores)}

Loading MiniLM Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
# Majority Baseline

def get_majority_scores(row):
    # Fixed scores based on training frequency
    # B=490, C=459, A=369, D=358, E=324
    return {'B': 5, 'C': 4, 'A': 3, 'D': 2, 'E': 1}

In [25]:
# ENSEMBLE 

def ensemble_predict(row, w1=0.50, w2=0.40, w3=0.10):
    """
    w1 = weight for DeBERTa
    w2 = weight for CrossEncoder  
    w3 = weight for Majority baseline
    """
    scores_deberta  = get_deberta_scores(row)
    scores_cross    = get_crossencoder_scores(row)
    scores_majority = get_majority_scores(row)

    # Normalize each model's scores to 0-1 range
    def normalize(d):
        vals = list(d.values())
        mn, mx = min(vals), max(vals)
        if mx == mn:
            return {k: 1.0 for k in d}
        return {k: (v - mn) / (mx - mn) for k, v in d.items()}

    s1 = normalize(scores_deberta)
    s2 = normalize(scores_cross)
    s3 = normalize(scores_majority)

    # Weighted sum
    final = {}
    for opt in ['A','B','C','D','E']:
        final[opt] = w1*s1[opt] + w2*s2[opt] + w3*s3[opt]

    ranked = sorted(final, key=final.get, reverse=True)
    return ' '.join(ranked[:3])

In [26]:

# VALIDATE ON TRAIN — find best weights

# Try different weight combinations
weight_configs = [
    (0.5,  0.4,  0.1),
    (0.6,  0.35, 0.05),
    (0.45, 0.50, 0.05),
    (0.7,  0.25, 0.05),
    (0.5,  0.5,  0.0),
]

best_map  = 0
best_weights = None

# Use 200 rows for quick weight search (saves time)
sample = train.sample(200, random_state=42)

for w1, w2, w3 in weight_configs:
    scores = []
    for _, row in sample.iterrows():
        pred = ensemble_predict(row, w1, w2, w3)
        scores.append(map_at_3(row['answer'], pred.split()))
    avg = round(sum(scores)/len(scores), 4)
    print(f"  Weights ({w1}, {w2}, {w3}) → MAP@3: {avg}")
    if avg > best_map:
        best_map = avg
        best_weights = (w1, w2, w3)

print(f"\nBest weights: {best_weights} → MAP@3: {best_map}")

  Weights (0.5, 0.4, 0.1) → MAP@3: 0.545
  Weights (0.6, 0.35, 0.05) → MAP@3: 0.5367
  Weights (0.45, 0.5, 0.05) → MAP@3: 0.515
  Weights (0.7, 0.25, 0.05) → MAP@3: 0.5442
  Weights (0.5, 0.5, 0.0) → MAP@3: 0.5258

Best weights: (0.5, 0.4, 0.1) → MAP@3: 0.545


In [27]:
# VALIDATE BEST CONFIG ON FULL TRAIN

w1, w2, w3 = best_weights
scores = []
for i, (_, row) in enumerate(train.iterrows()):
    pred = ensemble_predict(row, w1, w2, w3)
    scores.append(map_at_3(row['answer'], pred.split()))
    if i % 200 == 0:
        print(f"  [{i}/2000] MAP@3: {round(sum(scores)/len(scores), 4)}")

full_train_map = round(sum(scores)/len(scores), 4)
print(f"\nFinal Ensemble Train MAP@3: {full_train_map}")

  [0/2000] MAP@3: 0.5
  [200/2000] MAP@3: 0.5962
  [400/2000] MAP@3: 0.5827
  [600/2000] MAP@3: 0.5749
  [800/2000] MAP@3: 0.571
  [1000/2000] MAP@3: 0.5689
  [1200/2000] MAP@3: 0.5655
  [1400/2000] MAP@3: 0.56
  [1600/2000] MAP@3: 0.5613
  [1800/2000] MAP@3: 0.5566

Final Ensemble Train MAP@3: 0.5606


In [32]:
# GENERATE TEST SUBMISSION

predictions = []
for i, (_, row) in enumerate(test.iterrows()):
    pred = ensemble_predict(row, w1, w2, w3)
    predictions.append({'id': row['id'], 'prediction': pred})
    if i % 50 == 0:
        print(f"  {i}/{len(test)}")

submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)


  0/500
  50/500
  100/500
  150/500
  200/500
  250/500
  300/500
  350/500
  400/500
  450/500


In [ ]:
'''
Now this time , This submission uses a different approach where it uses an ensemble approach by combining multiple prediction methods.

Instead of relying on a single model, three different approaches are combined:

* DeBERTa Zero-Shot Classification
* MiniLM Cross-Encoder
* Majority Baseline

Each method gives a score to all five answer options. These scores are normalized and combined using different weights. 
The answer options are then ranked according to the final combined score, and the top three options are selected for submission.

The idea behind this approach is that each model has different strengths. By combining their predictions, the final result may become more accurate than using only one model.

By using this method, the score was : 0.47672 which is lesser than the Zero-Shot Classification model.


The Conclusion: Although ensemble methods generally help improve prediction accuracy,
This particular combination did not perform better than the previous submission.
As before that model achieved a score of 0.49958, whereas this ensemble scored 0.47672.

This suggests that simply adding more models does not always improve performance. 
The choice of models and the weights assigned to each model have a significant impact on the final results. 
Further tuning of the weights or selecting different models may help achieve better performance.

'''